# NL experiment · 02 The nighttime-light target of every cluster

This study is the IWI study of `2026_11_11_Wavelet` with another target: instead of the wealth (IWI) of a DHS cluster, the models predict the **average nighttime light (NL) of its patch** from Landsat and MODIS alone. The clusters, the files of `01_Download_Landsat_NL_MODIS`, the checks of `02_Check_Dataset`, the spatial folds of `03_Spatial_Folds` and the band caches are those of the IWI study and are read from its folder (`DATA_DIR`); nothing is downloaded again. Everything this study writes goes to `WaveletNL/` in that folder (`WORK_DIR`).

This notebook reads the `NL.tif` of every cluster of `train_clusters.csv` once and computes the target the way the IWI study normalised its NL input band (04–15): every pixel is clipped to `NL_CLIP` = 0–150 and normalised with `NormFunc` of the survey year, and the target is the mean of these values over the 224 × 224 pixels (6.72 km):

$$nl = \frac{1}{224^2}\sum_{pixels} \frac{\mathrm{clip}(NL, 0, 150) - Mean(year)}{stdDev(year)} = \frac{\overline{\mathrm{clip}(NL, 0, 150)} - Mean(year)}{stdDev(year)}$$

| period | NL image of 01 | `NormFunc` | target of a dark patch (NL = 0) |
|---|---|---|---|
| DMSP, surveys up to 2013 | DMSP-OLS `avg_vis`, DN 0–63 | Mean 4.153, stdDev 3.476 | −1.195 |
| VIIRS, surveys from 2014 | VIIRS VCMSLCFG `avg_rad`, nW cm⁻² sr⁻¹ | Mean 0.133, stdDev 1.748 | −0.076 |

**Columns added to `train_clusters.csv`** (written as `WORK_DIR/train_clusters_nl.csv`; 03 joins them to the folds):

| column | content |
|---|---|
| `nl` | **the target**: the mean of the normalised NL image (above) |
| `nl_raw` | the mean of the clipped NL image in the units of the sensor (DMSP DN or VIIRS radiance); `nl = (nl_raw − nl_norm_mean) / nl_norm_sd` |
| `nl_norm_mean`, `nl_norm_sd` | `NormFunc` of the survey year: a prediction in target units times `nl_norm_sd` plus `nl_norm_mean` is the patch mean in sensor units |
| `nl_raw_unclipped`, `nl_max` | the mean and the largest value of the NL image without clipping |
| `nl_dark_share`, `nl_clipped_share` | the share of pixels with NL ≤ 0 and with NL above the upper clip |
| `nl_period` | `DMSP` (year ≤ 2013) or `VIIRS` (from 2014): the `NormFunc` the target used; `nl_sensor` (from 02) is the sensor named in the file |

**Reading the target.** The normalisation is linear within each period, so an R² computed within the DMSP or within the VIIRS clusters is the same as in sensor units. Pooled over both periods, the target mixes two scales: a dark DMSP patch gets −1.195 and a dark VIIRS patch −0.076. The model notebooks therefore report the R² of each period next to the pooled one (`pooled R2 DMSP`, `pooled R2 VIIRS`), and 16 has a section per period. `NL_SENSORS = ['VIIRS']` (or `['DMSP']`) in 04–15 trains on one period only.

**Output** in `WORK_DIR`: `train_clusters_nl.csv`, `nl_target_stats.csv` (the target and `nl_raw` per period and survey year) and the plots of section 3. Nothing in `DATA_DIR` is changed.

**Running.** One pass over the `NL.tif` files with one process per core (a few minutes on a CPU). Then run `03_Folds_NL_Target`.

# Import libraries

In [ ]:
import os
import time
import multiprocessing

# The GPU image sets PROJ_LIB=/usr/share/proj, which has no proj.db, and rasterio then cannot read a CRS (as in 01-15).
for _var in ('PROJ_LIB', 'PROJ_DATA'):
    if os.environ.get(_var) and not os.path.exists(os.path.join(os.environ[_var], 'proj.db')):
        os.environ.pop(_var)

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display
try:
    from tqdm.auto import tqdm
except ImportError:                      # progress bars are optional
    def tqdm(it, **kw):
        return it

# Settings

In [ ]:
# ---- input / output ------------------------------------------------------------------------
DATA_DIR  = '/path/to/data/'   # the IWI study: Dataset/, its tables
                                                                             # (02, 03) and the band caches (04-15)
WORK_DIR  = os.path.join(DATA_DIR, 'WaveletNL')                  # this study: the target, the runs, the evaluation
IN_CSV    = os.path.join(DATA_DIR, 'train_clusters.csv')         # the usable clusters of 02_Check_Dataset (IWI study)
OUT_CSV   = os.path.join(WORK_DIR, 'train_clusters_nl.csv')      # IN_CSV + the NL target columns
STATS_CSV = os.path.join(WORK_DIR, 'nl_target_stats.csv')        # the target per period and survey year
NODATA    = -999999.0                                            # no-data value of the 01 files

# ---- the target: the NL image normalised as the NL input band of the IWI study (04-15), averaged over the patch.
# Nighttime light: clipped to NL_CLIP, then (NL - Mean) / stdDev with NormFunc of the survey year.
TARGET  = 'nl'
NL_CLIP = (0, 150)         # as in 26_tfrecords_generation_DMSP_VIIRS (VIIRS max 150, DMSP max 63); None: no clipping


def NormFunc(Year):
    if (Year >= 2014):     # VIIRS
        Mean = 0.1326654328849849
        stdDev = 1.7479698750873318
    else:                  # DMSP
        Mean = 4.152879625271373
        stdDev = 3.4762732210548317

    return Mean, stdDev


N_WORKERS = len(os.sched_getaffinity(0))   # the cores this job was given
os.makedirs(WORK_DIR, exist_ok=True)
print('clusters from', IN_CSV, '| output ->', WORK_DIR, '| workers:', N_WORKERS)

# 1. Clusters

The rows of `train_clusters.csv`: the clusters whose three files passed every check of `02_Check_Dataset` (so their `NL.tif` has no no-data pixel). The older column names are renamed as in 02 and 03 of the IWI study.

In [ ]:
clusters = pd.read_csv(IN_CSV, dtype={'cluster_id': str})
clusters = clusters.rename(columns={'ClusterID': 'cluster_id', 'year.of.interview': 'year',
                                    'month.of.interview': 'month', 'iwi_avg': 'iwi'})
assert clusters['cluster_id'].is_unique, f'a cluster_id appears twice in {IN_CSV}'
if 'folder' not in clusters:
    clusters['folder'] = [os.path.join(DATA_DIR, 'Dataset', str(c).replace('/', '_')) for c in clusters['cluster_id']]
if 'nl_sensor' not in clusters:                          # 01: DMSP up to 2013, VIIRS from 2014
    clusters['nl_sensor'] = np.where(clusters['year'] >= 2014, 'VIIRS', 'DMSP')
clusters['year'] = clusters['year'].astype(int)
clusters['nl_period'] = np.where(clusters['year'] >= 2014, 'VIIRS', 'DMSP')   # the NormFunc of the target
print(f"{len(clusters)} clusters, {clusters['country'].nunique()} countries, years "
      f"{clusters['year'].min()}-{clusters['year'].max()}")
print(pd.crosstab(clusters['nl_period'], clusters['nl_sensor'].fillna('?'), margins=True).to_string())

# 2. Read the NL image of every cluster

For every cluster: the `NL.tif` of 01 in float64, clipped to `NL_CLIP` and normalised with `NormFunc` of the survey year, as the NL input band of 04–15, then averaged over the patch. `nl_raw` is the same mean before the normalisation. A file that cannot be read, has another shape or holds a no-data pixel gets no target (`nl` is NaN, `nl_error` says why); 02 of the IWI study has already left such clusters out, so this should not happen.

In [ ]:
def nl_target_of(task):
    """The target and the descriptive columns of one cluster from its NL.tif (task: row position, folder, year)."""
    i, folder, year = task
    out = {'nl': np.nan, 'nl_raw': np.nan, 'nl_raw_unclipped': np.nan, 'nl_max': np.nan, 'nl_dark_share': np.nan,
           'nl_clipped_share': np.nan, 'nl_file_sensor': '', 'nl_error': ''}
    try:
        with rasterio.open(os.path.join(folder, 'NL.tif')) as src:
            a = src.read(1).astype(np.float64)
            out['nl_file_sensor'] = src.tags().get('sensor', '')
    except Exception as e:
        out['nl_error'] = f'unreadable: {type(e).__name__}: {e}'[:200]
        return i, out
    if a.shape != (224, 224):
        out['nl_error'] = f'shape {a.shape}'
        return i, out
    if (a == NODATA).any() or not np.isfinite(a).all():
        out['nl_error'] = 'no-data pixels'
        return i, out
    m, s = NormFunc(int(year))
    clipped = a if NL_CLIP is None else np.clip(a, *NL_CLIP)
    out.update({'nl': float(((clipped - m) / s).mean()),     # the mean of the normalised image, as 04-15 see it
                'nl_raw': float(clipped.mean()), 'nl_raw_unclipped': float(a.mean()), 'nl_max': float(a.max()),
                'nl_dark_share': float((a <= 0).mean()),
                'nl_clipped_share': float((a > NL_CLIP[1]).mean()) if NL_CLIP is not None else 0.0})
    return i, out


t0 = time.time()
tasks = [(i, f, y) for i, (f, y) in enumerate(zip(clusters['folder'], clusters['year']))]
found = [None] * len(tasks)
with multiprocessing.get_context('fork').Pool(N_WORKERS) as pool:
    for i, out in tqdm(pool.imap_unordered(nl_target_of, tasks, chunksize=16), total=len(tasks)):
        found[i] = out
target = pd.DataFrame(found, index=clusters.index)
params = [NormFunc(int(y)) for y in clusters['year']]
target.insert(2, 'nl_norm_mean', [m for m, _ in params])
target.insert(3, 'nl_norm_sd', [s for _, s in params])
print(f'read {len(target)} NL files in {time.time() - t0:.0f} s')

# the target is linear in nl_raw: check the identity on every cluster
ok = target['nl'].notna()
back = (target['nl_raw'] - target['nl_norm_mean']) / target['nl_norm_sd']
assert np.allclose(target.loc[ok, 'nl'], back[ok], rtol=0, atol=1e-9), 'nl and nl_raw do not agree'
print(f"target for {ok.sum()} of {len(target)} clusters; no target:",
      target.loc[~ok, 'nl_error'].str.split(':').str[0].value_counts().to_dict())
mismatch = (target['nl_file_sensor'] != '') & (target['nl_file_sensor'] != clusters['nl_period'])
if mismatch.any():
    print(f'WARNING: {mismatch.sum()} clusters whose NL file names another sensor than NormFunc of their year uses '
          '(NormFunc follows the year, as in 04-15):')
    display(clusters.loc[mismatch, ['cluster_id', 'year']].assign(sensor_in_file=target.loc[mismatch, 'nl_file_sensor']).head())

# 3. The target

Per period (the `NormFunc` the target used) and survey year: the number of clusters, the mean, standard deviation and percentiles of the target and of `nl_raw` (sensor units), and the share of dark patches (`nl_raw` = 0) and of patches with any lit pixel. The table goes to `nl_target_stats.csv`. The plots show the distribution of the target per period, `nl_raw` on a log scale, the target by survey year, and the target against the IWI of the cluster (the label of the IWI study, kept in the table), with their rank correlation per period.

In [ ]:
data = pd.concat([clusters, target], axis=1)
has = data[data[TARGET].notna()]


def describe(g):
    q = g[TARGET].quantile([0.1, 0.5, 0.9]).to_numpy()
    r = g['nl_raw'].quantile([0.5, 0.9, 0.99]).to_numpy()
    return pd.Series({'clusters': len(g), 'nl mean': g[TARGET].mean(), 'nl sd': g[TARGET].std(), 'nl min': g[TARGET].min(),
                      'nl p10': q[0], 'nl median': q[1], 'nl p90': q[2], 'nl max': g[TARGET].max(),
                      'nl_raw mean': g['nl_raw'].mean(), 'nl_raw median': r[0], 'nl_raw p90': r[1],
                      'nl_raw p99': r[2], 'dark patches %': 100 * (g['nl_raw'] <= 0).mean(),
                      'pixels clipped %': 100 * g['nl_clipped_share'].mean()})


COLS = [TARGET, 'nl_raw', 'nl_clipped_share']
by_period = has.groupby('nl_period')[COLS].apply(describe)
by_year = has.assign(year=has['year'].astype(str)).groupby(['nl_period', 'year'])[COLS].apply(describe)
stats = pd.concat([by_period.assign(year='all').set_index('year', append=True), by_year]).sort_index()
stats.to_csv(STATS_CSV)
print('->', STATS_CSV)
display(by_period.round(3))
by_year.round(3)

In [ ]:
PERIOD_COLOR = {'DMSP': '#eb6834', 'VIIRS': '#2a78d6'}
fig, ax = plt.subplots(1, 4, figsize=(21, 4.2), constrained_layout=True)
for p, g in has.groupby('nl_period'):
    ax[0].hist(g[TARGET], bins=60, alpha=0.6, color=PERIOD_COLOR.get(p), label=f'{p} (n = {len(g)})')
    ax[1].hist(np.log10(g['nl_raw'] + 0.01), bins=60, alpha=0.6, color=PERIOD_COLOR.get(p), label=p)
ax[0].set_yscale('log')
ax[0].set_xlabel(f'target {TARGET} (NormFunc units)')
ax[0].set_title('the target per period', loc='left')
ax[0].legend(frameon=False)
ax[1].set_xlabel('log10(nl_raw + 0.01), sensor units')
ax[1].set_title('patch mean of the clipped NL', loc='left')
years = sorted(has['year'].unique())
ax[2].boxplot([has.loc[has['year'] == y, TARGET] for y in years], positions=range(len(years)), widths=0.6,
              showfliers=False)
ax[2].set_xticks(range(len(years)))
ax[2].set_xticklabels(years, rotation=90)
ax[2].set_title('the target by survey year (DMSP up to 2013)', loc='left')
if 'iwi' in has:
    for p, g in has.groupby('nl_period'):
        ax[3].scatter(g['iwi'], g[TARGET], s=3, alpha=0.4, color=PERIOD_COLOR.get(p), label=p, rasterized=True)
    ax[3].set_xlabel('IWI (label of the IWI study)')
    ax[3].set_ylabel(TARGET)
    ax[3].set_title('the target against IWI', loc='left')
    ax[3].legend(frameon=False, markerscale=4)
for a in ax:
    a.grid(color='#e1e0d9', linewidth=0.8)
    a.set_axisbelow(True)
    for side in ('top', 'right'):
        a.spines[side].set_visible(False)
png = os.path.join(WORK_DIR, 'nl_target.png')
fig.savefig(png, dpi=150)
plt.show()
print('->', png)
if 'iwi' in has:
    corr = has.groupby('nl_period')[['iwi', TARGET]].apply(lambda g: pd.Series({
        'clusters': len(g), 'Spearman (iwi, nl)': g['iwi'].corr(g[TARGET], method='spearman'),
        'Pearson (iwi, nl)': g['iwi'].corr(g[TARGET])}))
    display(corr.round(3))

In [ ]:
# per country: clusters and the target per period, the share of dark patches and the rank correlation with IWI
def per_country(g):
    rec = {'clusters': len(g), 'dark patches %': 100 * (g['nl_raw'] <= 0).mean()}
    for p in ('DMSP', 'VIIRS'):
        gp = g[g['nl_period'] == p]
        rec[f'{p} clusters'] = len(gp)
        rec[f'{p} nl mean'] = gp[TARGET].mean() if len(gp) else np.nan
        rec[f'{p} nl_raw median'] = gp['nl_raw'].median() if len(gp) else np.nan
    if 'iwi' in g and len(g) > 2:
        rec['Spearman (iwi, nl)'] = g['iwi'].corr(g[TARGET], method='spearman')
    return pd.Series(rec)


(has.groupby('country')[[TARGET, 'nl_raw', 'nl_period'] + (['iwi'] if 'iwi' in has else [])].apply(per_country)
 .sort_values('clusters', ascending=False).round(3))

In [ ]:
# the NL image and the Landsat RGB of the darkest, a middle and the brightest cluster of every period
picks = []
for p, g in has.groupby('nl_period'):
    g = g.sort_values(TARGET)
    picks += [(p, 'darkest', g.index[0]), (p, 'median', g.index[len(g) // 2]), (p, 'brightest', g.index[-1])]
if picks:
    fig, axes = plt.subplots(2, len(picks), figsize=(3.3 * len(picks), 6.6), constrained_layout=True, squeeze=False)
    for k, (p, what, i) in enumerate(picks):
        folder = data.at[i, 'folder']
        with rasterio.open(os.path.join(folder, 'NL.tif')) as src:
            nl = src.read(1)
        with rasterio.open(os.path.join(folder, 'Landsat.tif')) as src:
            rgb = src.read([3, 2, 1]).astype(np.float32) * 0.0000275 - 0.2
        axes[0, k].imshow(np.moveaxis(np.clip(rgb / 0.3, 0, 1), 0, -1))
        axes[0, k].set_title(f"{p} {what}\n{data.at[i, 'cluster_id']} ({data.at[i, 'year']})", fontsize=8)
        im = axes[1, k].imshow(np.clip(nl, *NL_CLIP) if NL_CLIP else nl, cmap='magma')
        fig.colorbar(im, ax=axes[1, k], shrink=0.7)
        axes[1, k].set_title(f"nl {data.at[i, TARGET]:.2f}, nl_raw {data.at[i, 'nl_raw']:.2f}", fontsize=8)
        for a in axes[:, k]:
            a.set_xticks([])
            a.set_yticks([])
    plt.show()

# 4. Save

`train_clusters_nl.csv`: the rows of `train_clusters.csv` in their order with the target columns of section 2. It is written under a temporary name and then renamed. 03 joins it to the folds of the IWI study.

In [ ]:
cols = ['nl', 'nl_raw', 'nl_norm_mean', 'nl_norm_sd', 'nl_raw_unclipped', 'nl_max', 'nl_dark_share',
        'nl_clipped_share', 'nl_period', 'nl_file_sensor', 'nl_error']
out = pd.concat([clusters.drop(columns=[c for c in cols if c in clusters and c != 'nl_period']),
                 target], axis=1)
out = out[[c for c in out.columns if c not in cols] + cols]
tmp = OUT_CSV + '.tmp'
out.to_csv(tmp, index=False, float_format='%.10g')
os.replace(tmp, OUT_CSV)
print(f"{len(out)} clusters, {out[TARGET].notna().sum()} with a target ->", OUT_CSV)